In [1]:
# imports
import os
import csv
import shutil
import cv2
import string
import random
from torchvision import datasets, transforms
from torch.utils.data import Dataset
import numpy as np
import glob

# Data Sources
letters come from: https://www.kaggle.com/datasets/grassknoted/asl-alphabet

videos come from: https://huggingface.co/datasets/ZahidYasinMittha/American-Sign-Language-Dataset

# Images

In [2]:
# define input and output folders
images_source_root = "../images_folder"
images_output_root = os.path.join(os.getcwd(), "..", "letters")
images_output_root = os.path.abspath(images_output_root)

# define dataset split ratios
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# reproducability
random.seed(42)

# create output subfolders
splits = ["train", "val", "test"]
for split in splits:
    os.makedirs(os.path.join(images_output_root, split), exist_ok=True)

# loop through each class subfolder
for root, dirs, files in os.walk(images_source_root):
    label = os.path.basename(root)

    if label == os.path.basename(images_source_root): # skip folder itself
        continue

    # get images
    image_files = [f for f in files]

    # shuffle for random splitting
    random.shuffle(image_files)

    # split into train/test/val
    n_total = len(image_files)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    split_sets = {
        "train": image_files[:n_train],
        "val": image_files[n_train:n_train + n_val],
        "test": image_files[n_train + n_val:]
    }

    # copy images into structured folders
    for split_name, files_in_split in split_sets.items():
        split_label_folder = os.path.join(images_output_root, split_name, label)
        os.makedirs(split_label_folder, exist_ok=True)

        for idx, file in enumerate(files_in_split, start=1):
            image_id = f"{label.lower()}_{idx}"
            src_path = os.path.join(root, file)
            dst_path = os.path.join(split_label_folder, f"{image_id}.jpg")
            shutil.copy2(src_path, dst_path)


# Videos

In [3]:
def extract_frames(video_path, output_dir, num_frames=16):
    os.makedirs(output_dir, exist_ok=True)
    vid = cv2.VideoCapture(video_path)

    total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        vid.release()
        return

    # indices of frames to grab (evenly spaced)
    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)

    saved = 0
    for idx in indices:
        vid.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = vid.read()
        if not ret:
            continue
        frame_name = f"frame_{saved:04d}.jpg"
        cv2.imwrite(os.path.join(output_dir, frame_name), frame)
        saved += 1

    vid.release()
    cv2.destroyAllWindows()


In [4]:
# define input and output folders
words_source_root = "../hf_asl_videos"
words_output_root = os.path.join(os.getcwd(), "..", "words")
words_output_root = os.path.abspath(words_output_root)  # resolves ../ into a clean full path

# read and filter word list
all_words = [d for d in os.listdir(words_source_root) if os.path.isdir(os.path.join(words_source_root, d))]
alphabet = list(string.ascii_lowercase)
all_words = [w for w in all_words if w not in alphabet]
words_train = all_words

# create main output folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(words_output_root, split), exist_ok=True)

# traverse through dataset
for word in words_train:
    word_folder = os.path.join(words_source_root, word)
    if not os.path.exists(word_folder):
        continue

    # collect all video files for that word
    video_files = [os.path.join(word_folder, f) for f in os.listdir(word_folder) if f.lower().endswith((".mp4", ".mov"))]

    # shuffle for random splitting
    random.shuffle(video_files)

    # split into train/test/val
    n_total = len(video_files)
    n_train = int(train_ratio * n_total)
    n_val = int(val_ratio * n_total)

    split_sets = {
        "train": video_files[:n_train],
        "val": video_files[n_train:n_train + n_val],
        "test": video_files[n_train + n_val:]
    }

    # process and extract frames
    for split_name, split_videos in split_sets.items():
        for idx, video_path in enumerate(split_videos, start=1):
            video_id = f"{word}_{idx}"
            output_dir = os.path.join(words_output_root, split_name, word, video_id)
            os.makedirs(output_dir, exist_ok=True)
            extract_frames(video_path, output_dir)


# Test Data Loader in PyTorch

In [ ]:
# letters / images
letter_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

letter_train = datasets.ImageFolder(root='../letters/train', transform=letter_transform)
letter_val = datasets.ImageFolder(root='../letters/val', transform=letter_transform)
letter_test = datasets.ImageFolder(root='../letters/test', transform=letter_transform)

In [ ]:
# words / videos
class VideoFrameDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        for class_name in os.listdir(root):
            class_path = os.path.join(root, class_name)
            if not os.path.isdir(class_path):
                continue
            for video_folder in os.listdir(class_path):
                video_path = os.path.join(class_path, video_folder)
                frames = sorted(glob.glob(os.path.join(video_path, "*.jpg")))
                for frame in frames:
                    self.samples.append((frame, class_name))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


word_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

word_train = VideoFrameDataset(root=os.path.join(words_output_root, "train"), transform=word_transform)
word_val = VideoFrameDataset(root=os.path.join(words_output_root, "val"),   transform=word_transform)
word_test = VideoFrameDataset(root=os.path.join(words_output_root, "test"),  transform=word_transform)
